In [1]:
"""
Minimal Bayesian road-vs-background classifier for KITTI grayscale frames.

Assumptions
-----------
- Images are KITTI odometry grayscale frames (image_0) sized 1241x376.
- Road pixels appear mostly in a bottom trapezoid; background near top band.
- Uses 1D intensity histograms with Laplace smoothing to estimate likelihoods.

Usage
-----
    --data-root dataset/sequences/00/image_0 \
    --train-frames 80 \
    --save-frames 10 \
    --out-dir outputs/bayes_road

Outputs
-------
- /mask_XXXXXX.png     : binary MAP mask (road=255, bg=0)
- /overlay_XXXXXX.png  : red overlay of road mask on original frame
- /prob_XXXXXX.png     : grayscale probability map (0-255)
"""

#!/usr/bin/env python3
import argparse
import math
from pathlib import Path
import numpy as np
from PIL import Image
import cv2

def list_frames(data_root: Path) -> list[Path]:
    return sorted(data_root.glob("*.png"))

def get_trapezoid_mask(width: int, height: int, vp_y_rate=0.5, bottom_width_rate=0.9, top_width_rate=0.15):
    """
    소실점을 기준으로 도로 가능성이 높은 사다리꼴 마스크를 생성합니다.
    """
    mask = np.zeros((height, width), dtype=np.uint8)

    vp_y = int(height * vp_y_rate)  # 소실점 높이 (일반적으로 지평선 부근)

    # 사다리꼴 네 꼭짓점 정의
    p1 = [int(width * (0.5 - top_width_rate)), vp_y]
    p2 = [int(width * (0.5 + top_width_rate)), vp_y]
    p3 = [int(width * (0.5 + bottom_width_rate/2)), height]
    p4 = [int(width * (0.5 - bottom_width_rate/2)), height]

    pts = np.array([p1, p2, p3, p4], np.int32)
    cv2.fillPoly(mask, [pts], 1)
    return mask

def accumulate_weighted_histogram(img: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """마스크 영역 내의 픽셀만 사용하여 히스토그램 생성"""
    pixels = img[mask > 0]
    counts = np.bincount(pixels.flatten(), minlength=256)
    return counts.astype(np.float64)

def classify_frame(img: np.ndarray, road_p: np.ndarray, bg_p: np.ndarray, prior_road: float) -> tuple[np.ndarray, np.ndarray]:
    """Return MAP mask and probability map for road class."""
    eps = 1e-10  # numerical stability
    log_p_road = np.log(road_p[img] + eps) + math.log(prior_road)
    log_p_bg = np.log(bg_p[img] + eps) + math.log(1 - prior_road)

    logit = log_p_road - log_p_bg
    prob = 1.0 / (1.0 + np.exp(-np.clip(logit, -15, 15)))
    mask = logit > 0
    return mask, prob

def run(
    data_root: Path,
    train_frames: int,
    save_frames: int,
    out_dir: Path,
    prior_road: float,
    alpha: float = 0.8,
    video_path: Path | None = None,
    video_fps: float = 10.0,
    vp_y_rate: float = 0.5,
    bottom_width_rate: float = 0.9,
    top_width_rate: float = 0.15,
    max_frames: int | None = None,
    gif_path: Path | None = None,
    gif_fps: float = 8.0,
):
    """
    alpha: 시계열 일관성 계수 (0.8이면 기존 지식 80%, 새 프레임 20% 반영)
    save_frames: -1 이면 모든 프레임 저장
    video_path: 지정 시 overlay 프레임을 영상으로 기록
    """
    frames = list_frames(data_root)
    if not frames:
        raise SystemExit(f"No PNG frames found in {data_root}")

    sample_img = np.array(Image.open(frames[0]), dtype=np.uint8)
    h, w = sample_img.shape

    # 1. 소실점 기반 마스크 생성
    road_mask = get_trapezoid_mask(w, h, vp_y_rate, bottom_width_rate, top_width_rate)
    bg_mask = 1 - road_mask  # 도로 외 영역은 배경으로 간주

    # 초기 확률 분포 설정 (Laplace smoothing)
    road_p_total = np.ones(256)
    bg_p_total = np.ones(256)

    out_dir.mkdir(parents=True, exist_ok=True)

    limit = len(frames) if max_frames is None else min(max_frames, len(frames))
    max_frames_eval = max(train_frames, limit if save_frames >= 0 else len(frames))
    frames_iter = frames[:max_frames_eval]

    writer = None
    if video_path is not None:
        suffix = video_path.suffix.lower()
        if suffix == ".mp4":
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        else:  # default to MJPG/AVI
            fourcc = cv2.VideoWriter_fourcc(*"MJPG")
        writer = cv2.VideoWriter(str(video_path), fourcc, video_fps, (w, h))

    gif_frames: list[Image.Image] = []

    for idx, path in enumerate(frames_iter):
        img = np.array(Image.open(path), dtype=np.uint8)

        # 2. 현재 프레임의 히스토그램 추출
        curr_road_counts = accumulate_weighted_histogram(img, road_mask)
        curr_bg_counts = accumulate_weighted_histogram(img, bg_mask)

        # 3. 시계열 일관성 적용 (이동 평균)
        if idx == 0:
            road_p_total = curr_road_counts + 1
            bg_p_total = curr_bg_counts + 1
        else:
            # alpha 가중치를 이용해 이전 분포와 현재 분포를 결합
            road_p_total = alpha * road_p_total + (1 - alpha) * (curr_road_counts + 1)
            bg_p_total = alpha * bg_p_total + (1 - alpha) * (curr_bg_counts + 1)

        # 정규화하여 확률 밀도 함수 생성
        road_p = road_p_total / road_p_total.sum()
        bg_p = bg_p_total / bg_p_total.sum()

        # 4. 분류 및 저장 (save_frames 이내 혹은 영상 필요 시)
        should_save = (save_frames < 0 or idx < save_frames or writer is not None) and (max_frames is None or idx < max_frames)
        if should_save:
            mask, prob = classify_frame(img, road_p, bg_p, prior_road)

            stem = path.stem
            Image.fromarray((mask.astype(np.uint8) * 255)).save(out_dir / f"mask_{stem}.png")
            prob_img = np.clip(prob * 255, 0, 255).astype(np.uint8)
            Image.fromarray(prob_img).save(out_dir / f"prob_{stem}.png")

            overlay = np.stack([img, img, img], axis=-1)
            overlay[mask, 0] = 255  # 도로 영역 빨간색 강조
            overlay[mask, 1:] = (overlay[mask, 1:] * 0.3).astype(np.uint8)
            Image.fromarray(overlay).save(out_dir / f"overlay_{stem}.png")

            if writer is not None:
                writer.write(cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

            if gif_path is not None:
                gif_frames.append(Image.fromarray(overlay))

            print(f"Processed {stem} with temporal consistency")

    if writer is not None:
        writer.release()

    if gif_path is not None and gif_frames:
        duration_ms = int(1000 / gif_fps)
        gif_frames[0].save(
            gif_path,
            save_all=True,
            append_images=gif_frames[1:],
            duration=duration_ms,
            loop=0,
        )

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-root", type=Path, default=Path("D:/user/Downloads/data_odometry_gray/dataset/sequences/09/image_0"))
    parser.add_argument("--train-frames", type=int, default=80)
    parser.add_argument("--save-frames", type=int, default=10, help="frames to save; -1 for all")
    parser.add_argument("--out-dir", type=Path, default=Path("D:/user/Downloads/data_odometry_gray/outputs/bayes_road_advanced"))
    parser.add_argument("--prior-road", type=float, default=0.5)
    parser.add_argument("--alpha", type=float, default=0.85, help="Temporal consistency weight")
    parser.add_argument("--video-path", type=Path, default=None, help="if set, write overlay video here")
    parser.add_argument("--video-fps", type=float, default=10.0)
    parser.add_argument("--vp-y-rate", type=float, default=0.5, help="vanishing point height ratio")
    parser.add_argument("--bottom-width-rate", type=float, default=0.9)
    parser.add_argument("--top-width-rate", type=float, default=0.15)
    parser.add_argument("--max-frames", type=int, default=None, help="process at most this many frames (after sorting)")
    parser.add_argument("--gif-path", type=Path, default=None, help="optional GIF output of overlays")
    parser.add_argument("--gif-fps", type=float, default=8.0)
    args, _ = parser.parse_known_args()

    run(
        args.data_root,
        args.train_frames,
        args.save_frames,
        args.out_dir,
        args.prior_road,
        args.alpha,
        args.video_path,
        args.video_fps,
        args.vp_y_rate,
        args.bottom_width_rate,
        args.top_width_rate,
        args.max_frames,
        args.gif_path,
        args.gif_fps,
    )

Processed 000000 with temporal consistency
Processed 000001 with temporal consistency
Processed 000002 with temporal consistency
Processed 000003 with temporal consistency
Processed 000004 with temporal consistency
Processed 000005 with temporal consistency
Processed 000006 with temporal consistency
Processed 000007 with temporal consistency
Processed 000008 with temporal consistency
Processed 000009 with temporal consistency


## 문제 1. Projection Matrix 의미와 해석
Projection Matrix(투영 행렬)은 글로벌 좌표계의 3차원 점을 2차원 이미지 평면상에 투영하는 역할을 하며, 내부 행렬과 외부 행렬의 곱으로 구성되어 있습니다. 행렬의 형태는 3*4입니다.
### intrinsic 파라미터 (f_x, f_y, c_x, c_y)의 의미
내부 행렬은 카메라의 내적인 요인, 즉 초점 거리(focal length)와 광학 중심(optical center) 등의 파라미터 (f_x, f_y, c_x, c_y)로 표현됩니다.

카메라 렌즈의 중심과 이미지 센서 사이의 거리를 '유효 초점 거리'라고 합니다. 이 초점 거리에 이미지 센서의 픽셀 밀도를 곱하여 x축과 y축 방향에 대해 픽셀 단위로 표현한 f_x와 f_y를 산출합니다.

광학 중심(optical center이란 렌즈 중심을 관통하는 가상의 직선이 이미지 평면(image plane)과 수직으로 교차하는 지점을 말합니다. 이 교차 지점의 가로 위치와 세로 위치를 픽셀 단위로 표현한 값을 각각 c_x, c_y라고 합니다.

 ### extrinsic 파라미터 (R, t)의 역할
외부 요인은 외부 행렬 [R|t]이며 3x3 회전 행렬(R)과 3x1 이동 벡터(t)가 수평으로 결합된 구조를 갖습니다. 글로벌 좌표계에 있는 점을 해당 카메라의 좌표계로 변환하는 역할을 합니다.

회전 행렬(R)은 카메라의 방향을 표현하는 역할을 합니다. 각 행렬은 글로벌 좌표계 상에서 카메라의 축(x, y, z 방향)이 어느 방향을 나향하고 있는지를 나타냅니다.

이동 벡터(t)는 카메라의 위치를 표현하는 역할을 합니다. 글로벌 좌표계의 원점이 카메라 기준으로 어디에 위치하는지를 나타닙니다.


### Projection Matrix이 3D 점을 이미지 좌표로 변환하는 방식
- Projection Matrix이 3D 점을 이미지 좌표로 어떻게 변환하는지 수식으로 설명
#### 1. 3차원 공간에서 카메라 좌표계로의 변환
3차원 공간 벡터 수식은 다음과 같습니다.

$Xw = [xw, yw, zw]^T$

여기에 동차 좌표(Homogeneous Coordinate)를 추가하여 4차원으로 확장한 형태는 다음과 같습니다.

$Xw = [xw, yw, zw, 1]^T$

이제 외부 행렬(extrinsic matrix)에 동차 좌표를 추가한 값과 공간 벡터 수식을 곱해 카메라 좌표값($Xc$)를 구합니다.

$$\begin{bmatrix} x_c \\ y_c \\ z_c \\ 1 \end{bmatrix} = 
\begin{bmatrix} 
r_{11} & r_{12} & r_{13} & \bigm| & t_x \\ 
r_{21} & r_{22} & r_{23} & \bigm| & t_y \\ 
r_{31} & r_{32} & r_{33} & \bigm| & t_z \\ 
0 & 0 & 0 & \bigm| & 1 
\end{bmatrix} 
\begin{bmatrix} x_w \\ y_w \\ z_w \\ 1 \end{bmatrix}$$


#### 2. 카메라 좌표에서 이미지 픽셀 좌표로의 변환

계산한 카메라 좌표값($Xc$)과 내부 행렬(intrinsic matrix)을 통해 2차원 픽셀 좌표 $(u, v)$ 가 계산될 수 있습니다.

$$\begin{bmatrix} u \cdot s \\ v \cdot s \\ s \end{bmatrix} = 
\begin{bmatrix} 
f_x & 0 & c_x & 0 \\ 
0 & f_y & c_y & 0 \\ 
0 & 0 & 1 & 0 
\end{bmatrix}
\begin{bmatrix} x_c \\ y_c \\ z_c \\ 1 \end{bmatrix}$$

여기서 $s$는 카메라에서 해당 3D 점까지의 깊이($z_c$)을 의미합니다.

#### 3. 이미지 픽셀 좌표 최종 계산

$$u = f_x \cdot \frac{x_c}{z_c} + c_x$$

$$v = f_y \cdot \frac{y_c}{z_c} + c_y$$

